<a href="https://colab.research.google.com/github/asipnana/ProjectNLP/blob/main/notebooks/04_fasttext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone https://github.com/asipnana/ProjectNLP.git

Cloning into 'ProjectNLP'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 231 (delta 40), reused 6 (delta 6), pack-reused 137 (from 1)
Receiving objects: 100% (231/231), 1.83 MiB | 8.20 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [13]:
%cd /content/ProjectNLP

/content/ProjectNLP


In [1]:
!pip install numpy==1.26.4

In [2]:
!pip install fasttext

In [13]:
import fasttext
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)
from sklearn.metrics import classification_report

In [4]:
train_df = pd.read_csv("/content/ProjectNLP/dataset/processed/train (2).csv")
test_df = pd.read_csv("/content/ProjectNLP/dataset/processed/test (2).csv")

In [5]:
with open("fasttext_train.txt", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        label = row['sentiment']
        review = row['clean_review']
        f.write(f"__label__{label} {review}")

In [6]:
model = fasttext.train_supervised(input="fasttext_train.txt", epoch=25, lr=1.0, wordNgrams=2, dim=100)

In [7]:
y_true = []
y_pred = []

for _, row in test_df.iterrows():
    review = row['clean_review']
    prediction = model.predict(review)
    predicted_label = prediction[0][0]
    predicted_label = predicted_label.replace("__label__", "")
    y_true.append(row['sentiment'])
    y_pred.append(predicted_label)

In [12]:
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy: ", accuracy)

f1 = f1_score(y_true, y_pred, average="weighted")
print("F1-Score: ", f1)

Accuracy:  0.48115577889447236
F1-Score:  0.3126084195361882


In [14]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       413
    Positive       0.48      1.00      0.65       383

    accuracy                           0.48       796
   macro avg       0.24      0.50      0.32       796
weighted avg       0.23      0.48      0.31       796



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
results = pd.DataFrame({
    "Model": ["FastText"], "Accuracy": [accuracy_score(y_true, y_pred)],
    "Precision": [precision_score(y_true, y_pred, average="weighted")],
    "Recall": [recall_score(y_true, y_pred, average="weighted")],
    "F1": [f1_score(y_true, y_pred, average="weighted")]
})

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
results.to_csv("/content/ProjectNLP/results/fasttext_results.csv",index=False)

In [17]:
import os

os.makedirs("/content/ProjectNLP/models/fasttext",exist_ok=True)

In [18]:
model.save_model("/content/ProjectNLP/models/fasttext/fasttext_model.bin")